# Quickstart

This notebook provides a streamline pipeline for running calibration starting from processing raw target data. 

The pipeline here which abstracts/wraps away many of the detailed steps. For more transparency and explanation, please see the `walkthrough_*` notebooks for a tour of the major steps in the pipeline.

This pipeline assumes you have the following:

1. Raw ACS B19001 (household income) and B01001 (sex-by-age) tables for your CBSA downloaded from data.census.gov
2. Raw ATUS respondent, CPS, and activity files downloaded from BLS
3. Mobility sequences (per user-day) as a parquet file (with sequence columns + a `GEOID` home CBG column).

There are configs at the beginning that you will need to check/modify. All of the following steps will largely be automated based on these configs.

**TOC:**
- **0.** SETUP, PATHS, and CONFIGS (**THESE CONFIGS NEED TO BE MODIFIED ACCORDING TO YOUR NEEDS**).
- **1.** Process ACS (returns target CBG distributions and CBSA margins)
- **2.** Process ATUS (returns stratified metadata + behavioral clusters)
- **3.** Generate Mobility-ATUS cosine distance matrix
- **4.** K-NN cluster assignment and user validation
- **5.** Run calibration
- **6.** Inspect and save weights data


# 0. SETUP (START HERE)

## 0.1. Imports

In [1]:
from pathlib import Path

import pandas as pd
from collections import defaultdict

from mobcalibrate import Calibrator, CalibrationResult
from helpers import acs, atus, mobility
from helpers import prep_calibration_inputs as prep


## 0.2. CONFIGS: CBSA Code & Paths

In [2]:
CBSA_CODE = 38060

DATA_DIR = Path('data')
OUTPUT_DIR = Path('data/processed')

ACS_INCOME_FILE = DATA_DIR / f'acs/ACS5Y2020/{CBSA_CODE}/ACSDT5Y2020.B19001-Data.csv'
ACS_AGE_FILE    = DATA_DIR / f'acs/ACS5Y2020/{CBSA_CODE}/ACSDT5Y2020.B01001-Data.csv'

RESP_FILE = DATA_DIR / 'atus/atusresp-0324/atusresp_0324.dat'
CPS_FILE  = DATA_DIR / 'atus/atuscps-0324/atuscps_0324.dat'
ACT_FILE  = DATA_DIR / 'atus/atusact-0324/atusact_0324.dat'

MOBILITY_SEQ_FILE = DATA_DIR / 'mobility/user_day_sequences.parquet'


## 0.3. CONFIGS: ACS Processing

In [3]:
# Age group definitions 
# (if None, use acs.DEFAULT_AGE_GROUPS)
AGE_GROUPS = {
    "<18": [
        "Under 5 years", 
        "5 to 9 years",
        "10 to 14 years", 
        "15 to 17 years",
    ],

    "18-24": [
        "18 and 19 years", 
        "20 years",
        "21 years", 
        "22 to 24 years",
    ],

    "25-44": [
        "25 to 29 years", 
        "30 to 34 years",
        "35 to 39 years", 
        "40 to 44 years",
    ],

    "45-66": [
        "45 to 49 years", 
        "50 to 54 years",
        "55 to 59 years", 
        "60 and 61 years",
        "62 to 64 years", 
        "65 and 66 years",
    ],

    "67+": [
        "67 to 69 years", 
        "70 to 74 years",
        "75 to 79 years", 
        "80 to 84 years",
        "85 years and over",
    ],
}
DROP_AGE_GROUPS = ['<18']      # drop groups not represented in your mobility sample


# Income group mapping
# Leave None to auto-derive quartiles  from the CBSA-level row of the B19001 table.
# Otherwise, pass an explicit dict to override (see walkthrough_02_process_acs.ipynb for example).
INCOME_GROUP_MAPPING = None


## 0.4. CONFIGS: ATUS Processing

In [4]:
# years to pool data
YEARS = list(range(2004, 2020))
# minimum age of respondents
AGE_MIN = 18

# TEWHERE code -> activity label mapping, collapsing into categories that match
# SEQUENCE_METRIC_SPECS.all_labels below. 
"""
# Unlisted TEWHERE codes (e.g. transport) are mapped to 
# 'Unspecified place' with this defaultdict
TEWHERE_MAP = defaultdict(lambda: 'Unspecified place', {
    1:  'Home',
    2:  'Work',
    3:  'Unspecified place',  # someone else's home
    4:  'POI',                # restaurants/bars
    5:  'Unspecified place',  # place of worship
    6:  'POI',                # grocery store
    7:  'POI',                # other store/mall
    8:  'POI',                # school
    9:  'POI',                # outdoors
    10: 'Unspecified place',  # library
    11: 'Unspecified place',  # other place
    30: 'Unspecified place',  # bank
    31: 'POI',                # gym/health club
    32: 'Unspecified place',  # post office
    89: 'Unspecified place',
})
"""
TEWHERE_MAP = defaultdict(lambda: 'Other', {
    1:  'Home',
    2:  'Work'
})

# sequence cell width in minutes
T = 30

# number of behavioral clusters
K = 4

# respondent weight column name  
WEIGHT_COL = 'TUFINLWGT'

# stratification column names
ROW_VAR = 'income'
COL_VAR = 'age'

## 0.5. CONFIGS: Sequence Metrics & Distances

In [5]:
# specify alphabet labels present in the sequences
# as well as the subset of the metrics to use in distance calculations
SEQUENCE_METRIC_SPECS = {
    'home_label': 'Home',
    'work_label': 'Work',
    'all_labels': ['Home', 'Work', 'Other'],
    #'all_labels': ['Home', 'Work', 'Unspecified place', 'POI'],
    'feature_subset': 'all',   # 'all' | 'metrics_only' | 'tu_only' | 'edges_only'
}

# Column names in your mobility sequence DataFrame that contain the activity cells
# There should be 24*60/T of them (e.g. 48 cells for T=30 minutes).
MOB_SEQUENCE_COLS = [f'interval{n}' for n in range(int(24 * 60 / T))]

## 0.5. CONFIGS: Mobility Sequence Cluster Assignment

In [6]:
# number of nearest ATUS neighbors to find in the KNN cluster assignment
KNN_K = 10
# threshold for the number of neighbors in agreement 
# in order to assign a mobility user to cluster
KNN_THRESHOLD = 0.5

## 0.7. CONFIGS: Calibration

In [7]:
# number of replicates to run for variance estimation
# (each replicate resamples demographic attributes)
NUM_REPLICATES = 51

# seed number for calibration (used in sampling)
SEED = 1234

# calibration mode
# one of: 'uniform' | 'demographic' | 'behavioral' | 'demographic_behavioral'
CALIBRATION_MODE = 'demographic_behavioral'


# 1. Process ACS

Produces a CBG-level joint (income × age) distribution and CBSA-level margins. Income groups auto-derive as quartiles from the CBSA row (printed below); age groups come from `AGE_GROUPS` or the default.


In [8]:
acs_cbg_distr, income_margin, age_margin = acs.process_cbsa(
    ACS_INCOME_FILE, ACS_AGE_FILE,
    income_group_mapping=INCOME_GROUP_MAPPING,
    age_groups=AGE_GROUPS,
    drop_age_groups=DROP_AGE_GROUPS,
    row_var=ROW_VAR,
    col_var=COL_VAR,
)

print('acs_cbg_distr shape:', acs_cbg_distr.shape)
print('income margin:')
print(income_margin)
print('age margin:')
print(age_margin)
acs_cbg_distr.head()


acs.process_cbsa: auto-derived income quartile mapping: ['<35k', '35k-75k', '75k-125k', '125k+']
acs_cbg_distr shape: (2987, 9)
income margin:
     income       pop
0      <35k  416253.0
1   35k-75k  547471.0
2  75k-125k  410402.0
3     125k+  371093.0
age margin:
     age        pop
0  18-24   444363.0
1  25-44  1337601.0
2  45-66  1262610.0
3    67+   663574.0


,GEOID,18-24,25-44,45-66,67+,<35k,35k-75k,75k-125k,125k+
0,040130101021,0.003542,0.319953,0.279811,0.396694,0.085106,0.12766,0.378251,0.408983
1,040130101022,0.006592,0.071852,0.555043,0.366513,0.295756,0.087533,0.098143,0.518568
2,040130101023,0.023019,0.0,0.464151,0.51283,0.149068,0.10352,0.089027,0.658385
3,040130101031,0.045374,0.129004,0.55694,0.268683,0.22651,0.275168,0.20302,0.295302
4,040130101032,0.026382,0.236809,0.499372,0.237437,0.118758,0.140351,0.273954,0.466937


# 2. Process ATUS

Loads diaries, stratifies respondents against the ACS margins from Section 1, generates sequences and clusters them into K behavioral clusters via weighted k-medoids using `SEQUENCE_METRIC_SPECS`.


In [12]:
print('Loading ATUS diaries...')
resp, diaries = atus.load(
    RESP_FILE, CPS_FILE, ACT_FILE, TEWHERE_MAP,
    years=YEARS, age_min=AGE_MIN, cbsa_filter=[CBSA_CODE],
)

print('Stratifying respondents based on ACS targets...')
atus_meta, group_maps = atus.stratify(
    resp, income_margin, age_margin,
    row_var=ROW_VAR, col_var=COL_VAR,
)

print(f'Building sequences (cell width = {T} minutes)...')
atus_seq = atus.build_sequences(diaries, T=T, state_order=SEQUENCE_METRIC_SPECS['all_labels'])

print(f'Clustering sequences into {K} behavioural types...')
atus_metrics, cluster_results, medoid_thresholds = atus.cluster_sequences(
    atus_seq,
    weights=atus_meta[WEIGHT_COL],
    K=K,
    sequence_metric_specs=SEQUENCE_METRIC_SPECS,
    seed=SEED,
)

atus_meta['cluster_label'] = cluster_results['labels']
print('  cluster distribution:', atus_meta['cluster_label'].value_counts().sort_index().to_dict())

Loading ATUS diaries...
Stratifying respondents based on ACS targets...
auto-derived ATUS income groups: ['<35k', '35k-75k', '75k-125k', '125k+']
auto-derived ATUS age groups: ['18-24', '25-44', '45-66', '67+']
Building sequences (cell width = 30 minutes)...
Clustering sequences into 4 behavioural types...
  cluster distribution: {0: 262, 1: 714, 2: 494, 3: 563}


# 3. Build mobility-ATUS distance matrix

Uses `SEQUENCE_METRIC_SPECS` and `MOB_SEQUENCE_COLS` from config. 

`distance_to_atus` calculates the sequence metrics based on the spec, aligns the mobility metrics to atus metrics, then calculate cosine distance matrix based on the specified subset of the metrics.


In [ ]:
"""
mob_seq = pd.read_parquet(MOBILITY_SEQ_FILE)
display(mob_seq.head())

dist_df = mobility.distance_to_atus(
    mob_seq, atus_metrics,
    sequence_metric_specs=SEQUENCE_METRIC_SPECS,
    sequence_cols=MOB_SEQUENCE_COLS,
    geoid_col='GEOID',
)
print('dist_df shape:', dist_df.shape)
display(dist_df.head())
"""

In [13]:
# If already precomputed mobility-ATUS distance matrix, skip last step and load here:
#dist_df = pd.read_parquet(OUTPUT_DIR / f'dist_{CBSA_CODE}.parquet')

dist_df = pd.read_parquet(OUTPUT_DIR / f'distance_matrix_3cat_full_embedding.parquet')

# hypothetical user ids (since real ones cannot be published due to data agreement)
dist_df['user_id'] = range(1, len(dist_df)+1)

# fix bug in distance matrix
dist_df['20190504191857'] = dist_df.loc[:, '20190504191857'].str.replace('\x18', '').astype(float)

# 4. Cluster assignment + user filtering

Assigns each mobility user-day to one of K behavioural clusters via kNN voting with medoid-distance thresholds (users beyond the ATUS-derived 99th-percentile distance from their nearest medoid are left unassigned = -1), then drops any users whose home CBG is outside the ACS coverage.

In [14]:
assigned_labels = prep.assign_mobility_clusters(
    dist_df, atus_meta,
    cluster_label_col='cluster_label',
    k=KNN_K, threshold=KNN_THRESHOLD,
    medoid_indices=cluster_results['medoids'],
    medoid_thresholds=medoid_thresholds,
)

unit_ids, home_cbgs, assigned_labels, n_dropped = prep.filter_valid_users(
    dist_df.reset_index(),
    acs_cbg_distr,
    assigned_labels,
    geoid_col='GEOID',
    id_col='user_id',
)

print(f"Users retained: {len(home_cbgs)}")
print(f"Users dropped (missing CBG): {n_dropped}")
print("Proportion of assigned cluster labels:")
pd.Series(assigned_labels).value_counts(normalize=True).sort_index().rename("proportion")

100%|██████████| 141530/141530 [00:01<00:00, 95078.97it/s]


35151 rows have user GEOID missing from ACS GEOID (or null), e.g. ['040136103001' '040130506062' '040138171001' '040130405172'
 '040138118002']
Unique missing GEOIDs (excluding null): 504
Users retained: 106379
Users dropped (missing CBG): 35151
Proportion of assigned cluster labels:


-1    0.053770
 0    0.187979
 1    0.213463
 2    0.320684
 3    0.224104
Name: proportion, dtype: float64

# 5. Run calibration

This step preps the ACS marginal targets and the ATUS conditional cluster distribution, then instantiates a `Calibrator` object. 

Run `create_weights` in the chosen mode which returns a `CalibrationResult` with methods to get weights and metadata in various formats.


In [15]:
acs_targets = prep.prep_acs_targets(
    income_margin, 
    age_margin,
    row_var=ROW_VAR, 
    col_var=COL_VAR,
)

atus_target = prep.prep_atus_target(
    atus_meta,
    row_var=ROW_VAR, 
    col_var=COL_VAR,
    cluster_label_col='cluster_label',
    weight_col=WEIGHT_COL,
    num_row_cats=len(acs_targets['row_cats']),
    num_col_cats=len(acs_targets['col_cats']),
    num_clusters=K,
)

calibrator = Calibrator(
    unit_ids=unit_ids,
    home_cbgs=home_cbgs,
    assigned_cluster_labels=assigned_labels,
    acs_cbg_probs_df=acs_cbg_distr,
    acs_row_var_name=ROW_VAR,
    acs_col_var_name=COL_VAR,
    acs_row_cats=acs_targets['row_cats'],
    acs_col_cats=acs_targets['col_cats'],
    acs_row_margin=acs_targets['row_margin'],
    acs_col_margin=acs_targets['col_margin'],
    target_pop_tot=acs_targets['target_pop_tot'],
    atus_target_table=atus_target,
    num_replicates=NUM_REPLICATES,
    seed=SEED,
)

results = calibrator.create_weights(mode=CALIBRATION_MODE)

Weights created, returning CalibrationResult object.
Use all_final_weights() to obtain final weights.
Use to_df() to obtain replicate-specific stage1 and stage2 weights with metadata.
Use to_long_df() to obtain all replicate weights with metadata.



# 6. Inspect + save

A few quick sanity checks, plus two commented save options: a full pickle (everything) or a lighter long-format parquet (weights + metadata only).


In [16]:
print('main final weights sum:', int(results.all_final_weights()['weight_final_0'].sum()))
print('target population total:', results.target_pop_tot)
results.all_final_weights()

main final weights sum: 3704832
target population total: 3708148


,unit_id,weight_final_0,weight_final_1,weight_final_2,weight_final_3,weight_final_4,weight_final_5,weight_final_6,weight_final_7,weight_final_8,...,weight_final_41,weight_final_42,weight_final_43,weight_final_44,weight_final_45,weight_final_46,weight_final_47,weight_final_48,weight_final_49,weight_final_50
0,1,35,66,67,73,59,96,56,74,35,...,72,58,67,67,58,34,75,34,36,66
1,4,20,23,12,42,13,58,39,42,20,...,38,23,11,13,39,11,13,12,43,13
2,5,29,45,35,36,44,57,38,39,32,...,44,37,31,43,36,43,35,37,31,41
3,6,63,63,44,38,55,96,44,37,36,...,61,38,37,63,29,65,37,37,56,97
4,8,33,44,53,7,52,44,3,34,8,...,33,48,44,45,35,33,1,8,14,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106374,141526,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
106375,141527,63,30,66,29,43,37,56,58,63,...,37,57,34,36,38,45,29,57,43,29
106376,141528,12,38,12,15,20,24,16,24,20,...,20,12,42,16,38,15,12,11,58,24
106377,141529,24,15,15,12,43,20,12,15,43,...,20,15,15,25,59,25,24,11,12,26


In [17]:
print('first 5 rows of the main-replicate df:')
results.to_df(replicate_id=0).head()

first 5 rows of the main-replicate df:


,unit_id,weight1,weight_final,sampled_income_code,sampled_age_code,sampled_joint_stratum_code,calibration_mode
0,1,38,35,1,1,5,demographic_behavioral
1,4,35,20,1,2,6,demographic_behavioral
2,5,33,29,2,3,11,demographic_behavioral
3,6,31,63,3,3,15,demographic_behavioral
4,8,31,33,2,2,10,demographic_behavioral


In [18]:
print('first 5 rows of the all-replicate long df:')
results.to_long_df().head()

first 5 rows of the all-replicate long df:


,replicate_id,unit_id,weight1,weight_final,sampled_income_code,sampled_age_code,sampled_joint_stratum_code,calibration_mode
0,0,1,38,35,1,1,5,demographic_behavioral
1,0,4,35,20,1,2,6,demographic_behavioral
2,0,5,33,29,2,3,11,demographic_behavioral
3,0,6,31,63,3,3,15,demographic_behavioral
4,0,8,31,33,2,2,10,demographic_behavioral


In [ ]:
# Full pickle (all weights + full metadata as CalibrationResult object):
# results.save(OUTPUT_DIR / f'calibration_{CBSA_CODE}.pkl')

# Long-format parquet (weights from all replicates + metadata as long df):
# results.to_long_df().to_parquet(OUTPUT_DIR / f'weights_{CBSA_CODE}.parquet')